# 🚀 Salesfactory LinkedIn Automation - Das detaillierte Handbuch

Dieses Dokument ist deine zentrale Steuer- und Dokumentationszentrale. Es erklärt den kompletten Weg von einer Snov.io-Kontaktliste bis hin zur hochpersonalisierten, absendebereiten LinkedIn-Direktnachricht.

---

## 🛠️ 1. Das Setup (Einmalige Einrichtung)

Bevor du den Prozess das erste Mal startest, muss die Umgebung vorbereitet werden:

1. **Python & Virtuelle Umgebung:** Stelle sicher, dass Python installiert ist. Erstelle eine virtuelle Umgebung (`python -m venv .venv`) und aktiviere sie.
2. **Abhängigkeiten installieren:** Führe den Installations-Befehl aus (siehe Zelle unten), um alle nötigen Bibliotheken (`pandas`, `openpyxl`, `pyautogui`, `openai`, etc.) zu installieren.
3. **API-Key hinterlegen:** Öffne die `config.yaml` und trage unter `openrouter` deinen gültigen API-Key ein.
4. **Browser-Vorbereitung (Wichtig!):** Da das Skript lokales Web-Scraping über die Zwischenablage betreibt, **musst** du in deinem Standardbrowser bei LinkedIn eingeloggt sein. Der Browser sollte idealerweise schon geöffnet im Hintergrund liegen.

---

In [ ]:
!pip install pandas openpyxl pyautogui pyperclip openai pyyaml tqdm

## 📂 2. Was wird wo gemacht? (Architektur)

Das Projekt ist modular aufgebaut, damit Änderungen einfach vorgenommen werden können:

* **`config.yaml`:** Das Gehirn der Einstellungen. Hier liegen alle Pfade, Spaltennamen, Settings und Prompts. Du musst für Textänderungen nie in den Python-Code.
* **`tools/preprocessing.py`:** Nimmt deine dreckige Rohliste (z. B. aus Snov.io), bereinigt fehlerhafte URLs und generiert die Links für die LinkedIn-Aktivitäten-Seiten.
* **`tools/crawler.py`:** Ein physischer Bot. Er öffnet Browser-Tabs, drückt Strg+A und Strg+C, speichert den Text des LinkedIn-Profils und der Beiträge und schließt die Tabs wieder. 
* **`tools/api.py`:** Die Kommunikationsbrücke zu OpenRouter. Lädt die Config und sendet die Prompts + gecrawlten Text an das LLM (z. B. Qwen).
* **`run.py`:** Das Herzstück und der Orchestrator. Wenn der Agent startet, schickt er eine Meldung an WhatsApp, die den Online-Status mit Zeitstempel anzeigt. Danach liest er die vorbereitete Excel-Liste Zeile für Zeile ein, steuert den Crawler, sammelt die Ergebnisse, bewertet sie über das LLM, baut die Nachricht zusammen und speichert alles ab.

---

## 🔄 3. Der Standard-Workflow (Von Snov.io zur Kampagne)

So sieht dein täglicher Arbeitsablauf aus:

**Schritt 1: Export aus Snov.io**
Du exportierst deine recherchierte Liste (oder Kampagnen-Bounces) aus Snov.io als Excel-Datei. Nenne diese Datei `input.xlsx` und lege sie in den Hauptordner (`SALESFACTORY`). In dieser Liste steht irgendwo die LinkedIn-URL (meist in einer Spalte namens "LinkedIn" oder "Profil-URL").

**Schritt 2: Preprocessing (Datenaufbereitung)**
Du führst das Preprocessing-Skript aus. Es greift sich die `input.xlsx`, sucht die LinkedIn-URL, schneidet Tracking-Parameter (wie `?originalSubdomain=...`) ab und erstellt in der Spalte `User social` den passenden Link zur Aktivitätsseite (z.B. `.../recent-activity/all/`). Das Ergebnis wird als `contacts_urls.xlsx` gespeichert.

**Schritt 3: Crawling & LLM-Generierung**
Du startest die `run.py`. Das Skript geht nun jeden Kontakt der `contacts_urls.xlsx` durch:
1. Öffnet das Profil und die Beiträge im Browser und kopiert den Text (Dump).
2. Schickt den Dump ans LLM, um Punkte für Matching und Aktivität zu ermitteln.
3. Schickt den Dump ans LLM, um den "Firmenwinkel" (einen cleveren Einordnungssatz) zu extrahieren.
4. Baut aus Vorname, Firmenwinkel und Dump die hochpersonalisierte Einleitung.
5. Fügt den statischen Body (deinen Pitch) an.

*Achtung: Während der Crawler arbeitet, darfst du die Maus/Tastatur nicht benutzen, um die Zwischenablage nicht zu stören.*

**Schritt 4: Quality Gate & Export**
Nach den ersten 3 Kontakten pausiert das Skript und zeigt dir im Terminal eine Vorschau. Wenn die Qualität stimmt, drückst du Enter, und der Rest läuft durch. Am Ende hast du die `contacts_out.xlsx`, die du direkt für deinen Outreach (manuell oder per Automatisierungs-Tool) nutzen kannst.

---

## ⚙️ 4. Die Settings in der `config.yaml` erklärt

Unter dem Block `settings:` in der Config findest du wichtige Hebel:

* **`sleep_sec: 0.5`**: Die Pause in Sekunden zwischen zwei kompletten Kontakt-Durchläufen (schont die API und den Browser).
* **`overwrite: true`**: Steht dies auf `true`, werden bereits erstellte Nachrichten überschrieben. Steht es auf `false`, bearbeitet das Skript nur Zeilen, bei denen die Spalte "Vorlage" noch komplett leer ist (ideal, falls das Skript mal abstürzt und du weitermachen willst).
* **`enable_company_angle: true`**: Aktiviert den zusätzlichen LLM-Call, der versucht, die Herausforderungen des spezifischen Unternehmens aus dem Profil zu lesen, um den Einstieg noch relevanter zu machen.
* **`company_angle_cache: true`**: Speichert einmal generierte Firmenwinkel zwischen. Wenn du 5 Leute von der "Allianz" in der Liste hast, wird der Unternehmens-Satz nur beim ersten Mal generiert und bei den anderen 4 recycelt (spart Zeit und Token).
* **`enable_matching_score: true`**: Bewertet (0-100 Punkte), wie gut der Lead zu deiner Target-Audience (KI & Compliance) passt.
* **`enable_activity_score: true`**: Bewertet (0-100 Punkte), wie aktiv die Person auf LinkedIn ist (hilft dir zu filtern, wem du überhaupt schreiben solltest).

---

## 🧠 5. Die Prompts: Was macht welcher Text?

Unter dem Block `prompts:` definierst du die Persönlichkeit und Logik der KI:

* **`static_body`**: Dein "Brot und Butter"-Text. Das ist der Pitch, der immer exakt gleich ans Ende der generierten Einleitung geklebt wird. Hier steht dein Angebot und dein Name.
* **`system_guardrails_opening`**: Die harten, technischen Leitplanken für das LLM. Hier verbietest du Bullet Points, zwingst das Modell zu exakt 2 Sätzen im ersten Absatz und verbietest schlechte Angewohnheiten (wie "Ich habe Ihr Profil gesehen...").
* **`user_prompt_opening`**: Die inhaltliche Aufgabe. Hier legst du den Tonfall fest ("ruhig, nahbar, auf Augenhöhe"). Die "Fewshot Beispiele" sind extrem wichtig: Hier gibst du der KI konkrete Positiv- und Negativbeispiele, damit sie den Stil perfekt imitiert.
* **`company_angle_guardrails`**: Steuert die Generierung des optionalen Kontext-Satzes. Zwingt das Modell, keine Behauptungen aufzustellen, sondern nur Dinge zu nutzen, die im Dump stehen (um Peinlichkeiten zu vermeiden).
* **`matching_score_prompt` / `activity_score_prompt`**: Das sind strukturierte Bewertungsbögen. Du definierst hier genau, wofür es wie viele Punkte gibt (z.B. "20 Punkte, wenn genau 1 eigener Beitrag erkennbar ist"). Das LLM agiert hier als Analyst, nicht als Texter.

---

## ▶️ 6. Ausführung

Führe zuerst das Preprocessing aus, um die Snov.io-Liste (bzw. deine `input.xlsx`) vorzubereiten:

In [ ]:
!python tools/preprocessing.py

Starte danach den Haupt-Agenten. **Erinnerung:** Browser bereithalten und während des Durchlaufs Maus und Tastatur ruhen lassen!

In [ ]:
!python run.py